<a href="https://colab.research.google.com/github/Svatunek-Lab/teaching/blob/main/2026_University_of_Malaya/RDKit_CREST_Conformer_Comparison_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Conformer search: RDKit vs CREST
Generate and optimize conformers with RDKit, then inspect five supplied CREST geometries. **Energies from different methods are not directly comparable.**

In [ ]:
!pip -q install rdkit py3Dmol

In [ ]:
#@title RDKit conformer search (functions) { display-mode: "form" }
from rdkit import Chem
from rdkit.Chem import AllChem
import py3Dmol

def search_conformers(smiles, n=100, show=5):
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    if mol is None:
        raise ValueError('Invalid SMILES')
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    ids = list(AllChem.EmbedMultipleConfs(mol, numConfs=n, params=params))
    if not ids:
        raise ValueError('No conformers generated')
    if AllChem.MMFFHasAllMoleculeParams(mol):
        method = 'MMFF94'
        results = AllChem.MMFFOptimizeMoleculeConfs(mol, maxIters=400)
    elif AllChem.UFFHasAllMoleculeParams(mol):
        method = 'UFF'
        results = AllChem.UFFOptimizeMoleculeConfs(mol, maxIters=400)
    else:
        raise ValueError('No MMFF/UFF parameters available')
    ranked = sorted(zip(ids, results), key=lambda item: item[1][1])
    e0 = ranked[0][1][1]
    shown = ranked[:max(1, min(show, 12))]
    print(f'{len(ids)} generated conformers — {method} energies (kcal/mol)')
    for i, (cid, (status, energy)) in enumerate(shown, 1):
        print(f'{i}: ΔE = {energy-e0:.2f} kcal/mol' + (' [not converged]' if status else ''))
    cols = min(3, len(shown))
    rows = (len(shown)+cols-1)//cols
    view = py3Dmol.view(width=cols*290, height=rows*260, viewergrid=(rows, cols), linked=False)
    for i, (cid, (status, energy)) in enumerate(shown):
        panel = (i//cols, i%cols)
        view.addModel(Chem.MolToMolBlock(mol, confId=int(cid)), 'sdf', viewer=panel)
        view.setStyle({'stick':{'radius':0.16}, 'sphere':{'scale':0.24}}, viewer=panel)
        view.addLabel(f'RDKit #{i+1}: ΔE {energy-e0:.2f}',
                      {'position':{'x':0,'y':0,'z':0},'fontSize':12,'backgroundOpacity':0.55}, viewer=panel)
        view.zoomTo(viewer=panel)
    view.show()
    return mol, ranked

In [ ]:
#@title Generate RDKit conformers
smiles = "C[C@H]1[C@@H](Cl)CCCC1" #@param {type:"string"}
number_of_conformers = 100 #@param {type:"integer"}
show_lowest = 5 #@param {type:"integer"}

rdkit_mol, rdkit_ranked = search_conformers(smiles, number_of_conformers, show_lowest)

##Filter duplicate conformers (after optimization)
The first display shows the lowest-energy structures without duplicate filtering. Different starting geometries may optimize into the same minimum.

Here we compare heavy-atom RMSD after optimization, remove duplicates, and display the lowest-energy unique structures. Change the RMSD threshold and rerun to see how the results change.

In [ ]:
#@title Filter the optimized RDKit conformers
rmsd_threshold = 0.35 #@param {type:"number"}

from rdkit.Chem import rdMolAlign

if rmsd_threshold < 0:
    raise ValueError("RMSD threshold must not be negative")

# Work with heavy atoms only: methyl hydrogen rotations should not count.
heavy = Chem.RemoveHs(rdkit_mol)
unique = []

for cid, (status, energy) in rdkit_ranked:
    if all(rdMolAlign.GetBestRMS(heavy, heavy, prbId=cid, refId=other)
           >= rmsd_threshold for other, _ in unique):
        unique.append((cid, energy))

print(f"Before filtering: {len(rdkit_ranked)} conformers")
print(f"After filtering:  {len(unique)} unique conformers")
print(f"Duplicates removed: {len(rdkit_ranked) - len(unique)}")

shown = unique[:show_lowest]
if shown:
    e0 = rdkit_ranked[0][1][1]
    cols = min(3, len(shown))
    rows = (len(shown) + cols - 1) // cols
    view = py3Dmol.view(width=cols*290, height=rows*260,
                        viewergrid=(rows, cols), linked=False)
    for i, (cid, energy) in enumerate(shown):
        panel = (i//cols, i%cols)
        view.addModel(Chem.MolToMolBlock(rdkit_mol, confId=int(cid)), 'sdf', viewer=panel)
        view.setStyle({'stick': {'radius': 0.16}, 'sphere': {'scale': 0.24}}, viewer=panel)
        view.addLabel(f'Unique #{i+1}: ΔE {energy-e0:.2f}',
                      {'position': {'x': 0, 'y': 0, 'z': 0}, 'fontSize': 12,
                       'backgroundOpacity': 0.55}, viewer=panel)
        view.zoomTo(viewer=panel)
    view.show()


## Five CREST conformers
Geometries supplied as a concatenated XYZ file, in the provided order. **No CREST energies were supplied.**

In [ ]:
#@title Display supplied CREST conformers { display-mode: "form" }
import py3Dmol

crest_xyz = '21\nGenerated by ChemView\nC -3.963291 0.548731 -0.000297\nC -2.439934 0.533315 0.015580\nC -1.863860 1.949423 0.028478\nC -2.435206 2.743855 1.203633\nC -3.959480 2.779111 1.145394\nC -4.522758 1.361863 1.163779\nH -2.054293 -0.021826 -0.844940\nH -4.285406 0.991244 -0.946768\nH -4.342093 -0.474383 0.039946\nH -2.116491 2.278276 2.137890\nH -2.036688 3.759949 1.180675\nH -4.348855 3.343210 1.994848\nH -4.278117 3.287009 0.231321\nH -4.251275 0.876647 2.102830\nH -5.612068 1.387712 1.099917\nH -2.197640 2.421254 -0.904544\nC -0.338044 1.942770 0.046644\nH 0.053410 1.383106 -0.800418\nH 0.041981 2.960189 -0.006441\nH 0.029521 1.485189 0.961583\nCl -1.879389 -0.389726 1.447340\n21\nGenerated by ChemView\nC -3.545456 0.321362 0.540590\nC -2.677298 0.847016 -0.591711\nC -1.784655 2.013474 -0.163585\nC -2.690909 3.123517 0.388155\nC -3.586863 2.631830 1.521978\nC -4.433696 1.448721 1.064147\nH -3.315807 1.170902 -1.419902\nH -4.161292 -0.502954 0.175716\nH -2.915115 -0.063826 1.342765\nH -2.070152 3.950289 0.737862\nH -3.320797 3.500256 -0.421674\nH -2.977283 2.336248 2.376774\nH -4.236916 3.446241 1.848009\nH -5.042471 1.080048 1.891502\nH -5.109675 1.773233 0.269265\nH -1.282024 2.396005 -1.059873\nC -0.698501 1.612699 0.832296\nH -0.068471 0.835770 0.407204\nH -0.074900 2.474471 1.058798\nH -1.118631 1.241388 1.761932\nCl -1.689061 -0.489773 -1.263797\n21\nGenerated by ChemView\nC -3.566208 0.339995 0.527761\nC -2.651093 0.834632 -0.581064\nC -1.756706 1.994663 -0.137607\nC -2.669885 3.123481 0.363353\nC -3.602890 2.654551 1.476246\nC -4.458402 1.486721 0.998095\nH -3.252482 1.150125 -1.439136\nH -4.174593 -0.487678 0.158845\nH -2.970341 -0.027898 1.364260\nH -2.052815 3.951984 0.715593\nH -3.271737 3.489724 -0.472059\nH -3.019585 2.341481 2.343709\nH -4.243003 3.482418 1.787518\nH -5.106079 1.139426 1.804747\nH -5.095339 1.814114 0.172910\nH -1.231745 2.360318 -1.026482\nC -0.710474 1.583233 0.910102\nH -0.807049 2.176245 1.815602\nH -0.807843 0.535277 1.178494\nH 0.294776 1.726793 0.523268\nCl -1.646395 -0.522574 -1.187703\n21\nGenerated by ChemView\nC -3.935184 0.388059 0.209688\nC -2.407887 0.478778 0.277682\nC -1.917863 1.898659 0.011399\nC -2.435689 2.861861 1.090219\nC -3.827920 2.469463 1.600812\nC -4.608053 1.729095 0.520786\nH -1.952235 -0.219591 -0.429248\nH -4.223218 0.057563 -0.790898\nH -4.278297 -0.369922 0.915312\nH -1.735607 2.872165 1.926602\nH -2.460366 3.868792 0.669316\nH -3.735160 1.818293 2.470759\nH -4.372220 3.362213 1.912648\nH -5.633030 1.552029 0.849542\nH -4.652357 2.346100 -0.378901\nH -2.344969 2.184209 -0.957052\nC -0.397840 1.966723 -0.100901\nH -0.081881 2.991025 -0.284044\nH 0.066703 1.621564 0.819630\nH -0.044023 1.344606 -0.920673\nCl -1.822879 -0.074766 1.883771\n21\nGenerated by ChemView\nC -3.827488 0.358768 0.688891\nC -2.425883 0.460677 0.079898\nC -1.918621 1.909144 0.047837\nC -2.431123 2.646084 1.284346\nC -3.955454 2.809057 1.220452\nC -4.612128 1.647765 0.464029\nH -2.403112 0.041529 -0.929515\nH -4.355727 -0.493131 0.257760\nH -3.744856 0.186047 1.763208\nH -2.145141 2.075388 2.169413\nH -1.956052 3.625092 1.358078\nH -4.352938 2.860056 2.235320\nH -4.208559 3.747410 0.723693\nH -5.639120 1.514517 0.806548\nH -4.651112 1.862962 -0.605484\nH -2.362849 2.374895 -0.839428\nC -0.402224 2.007970 -0.089065\nH -0.113502 3.042136 -0.262949\nH 0.090590 1.660317 0.814751\nH -0.053228 1.406914 -0.925968\nCl -1.331449 -0.596678 1.034557'

lines = crest_xyz.strip().splitlines()
structures = []
while lines:
    count = int(lines[0])
    structures.append("\n".join(lines[:count+2]) + "\n")
    lines = lines[count+2:]

cols = 3
rows = (len(structures)+cols-1)//cols
viewer = py3Dmol.view(width=870, height=rows*270, viewergrid=(rows,cols), linked=False)
for i, structure in enumerate(structures):
    panel = (i//cols,i%cols)
    viewer.addModel(structure,"xyz",viewer=panel)
    viewer.setStyle({"stick":{"radius":0.16},"sphere":{"scale":0.24}},viewer=panel)
    viewer.addLabel(f"CREST #{i+1}",{"position":{"x":-3,"y":4,"z":0},"fontSize":14,"backgroundOpacity":0.5},viewer=panel)
    viewer.zoomTo(viewer=panel)
viewer.show()